# NB8 — System Preference Plots

In [ ]:
import os
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

import pandas as pd
import numpy as np
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from itertools import combinations

TABLES = "outputs/tables/"
FOLDER = "outputs/graphs/NB8_System_Preference/"

def _set_font():
    tnr=[f.name for f in fm.fontManager.ttflist if "Times New Roman" in f.name]
    plt.rcParams["font.serif"]=([ "Times New Roman"] if tnr else [])+["Liberation Serif","DejaVu Serif"]
    plt.rcParams["font.family"]="serif"
_set_font()
plt.rcParams.update({"font.size":10,"axes.titlesize":11,"axes.labelsize":10,
    "xtick.labelsize":9,"ytick.labelsize":9,"legend.fontsize":9,"figure.dpi":150})

RATER_ORDER=["Human 1","Human 2","Sarvam-105B","LLaMA-4-Scout-17B","GPT-Oss-120B","Gemini-3.1-Flash-Lite-Preview"]
# Updated to include Holistic Total
TASK_ORDER =["Detection (C1)","Span+Description (C2)","Category (C3)","Correction (C4)","Holistic Total (Out of 6)"]
CAT_ORDER  =["Script Norm.","Spelling","Grammatical","Code-Mixing","Correct"]
GROUP_ORDER=[
    "G1 — Humans",
    "G2 — Annotator LLMs",
    "G3 — Non-Annotator LLMs",
    "G4 — All LLMs",
    "G5 — Humans + Annotator LLMs",
    "G6 — Humans + Non-Annotator LLMs",
    "G7 — All Raters",
]

COLORS5=["#4472C4","#ED7D31","#70AD47","#9E480E","#7030A0"]
COLORS2=["#4472C4","#ED7D31"]

def ab(ax,bars,vals,fmt=".2f",fs=7):
    ylim=ax.get_ylim(); yr=ylim[1]-ylim[0]; off=yr*0.018
    for bar,val in zip(bars,vals):
        try: v=float(val)
        except: continue
        if math.isnan(v): continue
        y=bar.get_height()
        yo=np.clip(y+off if v>=0 else y-off*2,ylim[0]+yr*0.01,ylim[1]-yr*0.05)
        ax.text(bar.get_x()+bar.get_width()/2,yo,f"{v:{fmt}}",
                ha="center",va="bottom" if v>=0 else "top",fontsize=fs,fontweight="normal")

def save_fig(fig,folder,filename):
    os.makedirs(folder,exist_ok=True)
    fig.savefig(os.path.join(folder,filename+".svg"),format="svg",bbox_inches="tight")
    fig.savefig(os.path.join(folder,filename+".pdf"),format="pdf",bbox_inches="tight")
    plt.close(fig); print("Saved:",filename)

print("Setup complete.")

Mounted at /content/drive
Setup complete.


In [ ]:
agree_df = pd.read_excel(TABLES+"NB8_Agreement_Rate.xlsx")
multi_df = pd.read_excel(TABLES+"NB8_Multi_Rater_Metrics.xlsx")
pair_df  = pd.read_excel(TABLES+"NB8_Pairwise_Cohen_Kappa.xlsx")
bias_df  = pd.read_excel(TABLES+"NB8_Optimism_Bias.xlsx")
print("Loaded all NB8 result tables.")

Loaded all NB8 result tables.


## Plot 1: Overall Agreement Rate Heatmap

In [ ]:
# ── Plot 1: Agreement Rate heatmap — evaluator × task (overall) ─
ov=agree_df[agree_df["Cat_Short"]=="ALL"].copy()
pivot=ov.pivot(index="Rater_Label",columns="Task_Label",values="Agreement_Rate")
pivot=(pivot.reindex(index=RATER_ORDER,columns=TASK_ORDER)*100).round(1)
fig,ax=plt.subplots(figsize=(11,6))
sns.heatmap(pivot,annot=True,fmt=".1f",cmap="RdYlGn",vmin=0,vmax=100,
            linewidths=0.5,ax=ax,annot_kws={"size":10},square=True)
ax.set_title("Agreement Rate (%) with Human Gold\n(S/L/T preference labels, N=50 sentences per task)",fontsize=11)
ax.set_xlabel("Task"); ax.set_ylabel("Evaluator")
ax.set_xticklabels(ax.get_xticklabels(),rotation=30,ha="right",fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(),rotation=0,fontsize=9)
plt.tight_layout()
save_fig(fig,FOLDER,"01_agreement_rate_overall_heatmap")

Saved: 01_agreement_rate_overall_heatmap


## Plot 2: Agreement Rate by Category and Task

In [ ]:
# ── Plot 2: Agreement rate by evaluator × category, task panels ─
# One panel per task (now 5 tasks including Total)
fig,axes=plt.subplots(1,5,figsize=(32,7))
for ai,task_label in enumerate(TASK_ORDER):
    ax=axes[ai]
    sub=agree_df[(agree_df["Task_Label"]==task_label)&(agree_df["Cat_Short"]!="ALL")].copy()
    pivot=sub.pivot(index="Rater_Label",columns="Cat_Short",values="Agreement_Rate")
    pivot=(pivot.reindex(index=RATER_ORDER,columns=CAT_ORDER)*100).round(1)
    sns.heatmap(pivot,annot=True,fmt=".1f",cmap="RdYlGn",vmin=0,vmax=100,
                linewidths=0.5,ax=ax,annot_kws={"size":8},square=True)
    ax.set_title(task_label,fontsize=10)
    ax.set_xlabel("Error Category",fontsize=9)
    ax.set_ylabel("Evaluator" if ai==0 else "",fontsize=9)
    ax.set_xticklabels(ax.get_xticklabels(),rotation=35,ha="right",fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(),rotation=0,fontsize=8)
plt.suptitle("Agreement Rate (%) with Human Gold by Evaluator, Category and Task",y=1.02,fontsize=12)
plt.tight_layout()
save_fig(fig,FOLDER,"02_agreement_rate_by_category_task")

Saved: 02_agreement_rate_by_category_task


## Plot 3: Multi-Rater Chance-Corrected Agreement

In [ ]:
# ── Plot 3: Multi-rater metrics — Fleiss κ, Krippendorff α, Gwet AC₁ ─
# ab3: staggered heights per task index + vertical text to avoid overlap on narrow bars
def ab3(ax,bars,vals,ti,n_tasks,fmt=".2f",fs=5):
    ylim=ax.get_ylim(); yr=ylim[1]-ylim[0]
    base_off=yr*0.02
    stagger=base_off*ti  # each task shifted up slightly
    for bar,val in zip(bars,vals):
        try: v=float(val)
        except: continue
        if math.isnan(v): continue
        y=bar.get_height()
        yo=np.clip(y+base_off+stagger if v>=0 else y-base_off-stagger,
                   ylim[0]+yr*0.01,ylim[1]-yr*0.06)
        ax.text(bar.get_x()+bar.get_width()/2,yo,f"{v:{fmt}}",
                ha="center",va="bottom",fontsize=fs,fontweight="normal",rotation=90)
ov_multi=multi_df[multi_df["Cat_Short"].isna()].copy()
metrics=[("Fleiss_Kappa","Fleiss' \u03ba"),
         ("Kripp_Alpha_Nominal","Krippendorff's \u03b1"),
         ("Gwet_AC1","Gwet's AC$_1$")]
fig,axes=plt.subplots(1,3,figsize=(20,7))
for mi,(col,label) in enumerate(metrics):
    ax=axes[mi]; w=0.15; x=list(range(len(GROUP_ORDER)))
    for ti,task in enumerate(TASK_ORDER):
        sub=ov_multi[ov_multi["Task_Label"]==task].set_index("Group").reindex(GROUP_ORDER)
        vals=sub[col].tolist()
        bars=ax.bar([xi+ti*w for xi in x],vals,w,label=task,color=COLORS5[ti])
        ab3(ax,bars,vals,ti,len(TASK_ORDER),fmt=".2f",fs=5)
    ax.set_xticks([xi+2*w for xi in x])
    ax.set_xticklabels(GROUP_ORDER,fontsize=8,rotation=15,ha="right")
    ax.set_ylabel(label); ax.set_title(f"{label} by Rater Group and Task",fontsize=10)
    ax.legend(fontsize=7,loc="upper right"); ax.axhline(0,color="black",lw=0.8,ls="--")
    ax.set_ylim(-0.35,1.2)
plt.suptitle("Chance-Corrected Multi-Rater Agreement on S/L/T Preference Labels (N=50)",y=1.02,fontsize=12)
plt.tight_layout()
save_fig(fig,FOLDER,"03_multi_rater_metrics_by_group")

Saved: 03_multi_rater_metrics_by_group


## Plot 4: Pairwise Cohen's κ Heatmaps

In [ ]:
# ── Plot 4: Pairwise Cohen's κ heatmap, task panels ──────────
rn=RATER_ORDER
fig,axes=plt.subplots(1,5,figsize=(32,8))
for ai,task_label in enumerate(TASK_ORDER):
    ax=axes[ai]
    mat=pd.DataFrame(np.nan,index=rn,columns=rn)
    subset=pair_df[(pair_df["Task_Label"]==task_label)&(pair_df["Cat_Short"].isna())]
    for _,row in subset.iterrows():
        r1=row["Rater1_Label"]; r2=row["Rater2_Label"]; k=row["Cohen_Kappa"]
        mat.loc[r1,r2]=k; mat.loc[r2,r1]=k
    m2=mat.values.copy().astype(float); np.fill_diagonal(m2,1.0)
    mat=pd.DataFrame(m2,index=rn,columns=rn)
    sns.heatmap(mat,annot=True,fmt=".2f",cmap="coolwarm",vmin=-0.2,vmax=1.0,
                linewidths=0.5,ax=ax,annot_kws={"size":7},square=True)
    ax.set_title(f"{task_label}\nPairwise Cohen's \u03ba",fontsize=10)
    ax.set_xticklabels(ax.get_xticklabels(),rotation=45,ha="right",fontsize=7)
    ax.set_yticklabels(ax.get_yticklabels(),rotation=0,fontsize=7)
plt.suptitle("Pairwise Cohen's \u03ba on S/L/T Preference Labels (N=50 sentences)",y=1.02,fontsize=12)
plt.tight_layout()
save_fig(fig,FOLDER,"04_pairwise_cohen_kappa")

Saved: 04_pairwise_cohen_kappa


## Plot 5: Optimism Bias

In [ ]:
# ── Plot 5: Optimism bias by category and task ────────────────
fig,axes=plt.subplots(1,5,figsize=(22,5))
for ai,task_label in enumerate(TASK_ORDER):
    ax=axes[ai]
    sub=bias_df[bias_df["Task_Label"]==task_label]
    for ri,(rater,color) in enumerate([("Sarvam-105B",COLORS2[0]),("LLaMA-4-Scout-17B",COLORS2[1])]):
        rd=sub[sub["Rater"]==rater].set_index("Cat_Short").reindex(CAT_ORDER)
        vals=rd["Optimism_Bias"].tolist()
        x_pos=[xi+ri*0.38 for xi in range(len(CAT_ORDER))]
        bars=ax.bar(x_pos,vals,0.36,label=rater,color=color)
        ab(ax,bars,vals,fmt=".2f",fs=7)
    ax.set_xticks([xi+0.19 for xi in range(len(CAT_ORDER))])
    ax.set_xticklabels(CAT_ORDER,fontsize=8)
    ax.set_title(task_label,fontsize=10); ax.set_ylabel("Optimism Bias")
    ax.axhline(0,color="black",lw=0.8,ls="--"); ax.set_ylim(-0.6,1.1)
    if ai==0: ax.legend(fontsize=8)
plt.suptitle("Optimism Bias of Self-Evaluating LLMs by Category and Task\n"
             "(positive = more lenient toward own outputs than human baseline)",y=1.03,fontsize=12)
plt.tight_layout()
save_fig(fig,FOLDER,"05_optimism_bias_by_category")

/tmp/ipykernel_1210/2166283178.py:19: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Saved: 05_optimism_bias_by_category


## Plot 6: Gwet's AC₁ Summary Heatmap

In [ ]:
# ── Plot 6: Gwet AC₁ heatmap — group × task ─────────────────
ov6=multi_df[multi_df["Cat_Short"].isna()].copy()
pivot6=ov6.pivot(index="Group",columns="Task_Label",values="Gwet_AC1")
pivot6=pivot6.reindex(index=GROUP_ORDER,columns=TASK_ORDER)
fig,ax=plt.subplots(figsize=(11,5))
sns.heatmap(pivot6,annot=True,fmt=".3f",cmap="RdYlGn",vmin=-0.2,vmax=1.0,
            linewidths=0.5,ax=ax,annot_kws={"size":10},square=True)
ax.set_title("Gwet's AC$_1$ by Rater Group and Task\n(S/L/T preference labels, N=50 sentences)",fontsize=11)
ax.set_xlabel("Task"); ax.set_ylabel("Rater Group")
ax.set_xticklabels(ax.get_xticklabels(),rotation=30,ha="right",fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(),rotation=0,fontsize=9)
plt.tight_layout()
save_fig(fig,FOLDER,"06_gwet_ac1_heatmap_group_task")

Saved: 06_gwet_ac1_heatmap_group_task
